### 1) 라이브러리 설치

In [ ]:
# uv add pypdf=">=4.2.0,<5.0.0"
# uv add langchain-upstage langchain-community langchain-text-splitters faiss-cpu python-dotenv

#### PyPDFLoader 간단한 예제

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# ChatUpstage, UpstageEmbeddings 가 UPSTAGE_API_KEY 환경 변수를 자동으로 읽음 (키 값은 출력하지 않음)
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

UPSTAGE_API_KEY 설정 확인 완료


#### Upstage 를 사용한 RAG
* ChatUpstage
* UpstageEmbeddings
* PyPDFLoader (PDF 텍스트 추출)

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings

print("==> 1. 문서 로딩 → PDF 읽기...")
PDF_PATH = "../data/tutorial-korean.pdf"
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"  총 {len(documents)}페이지 로드 완료")


c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


==> 1. 문서 로딩 → PDF 읽기...
  총 39페이지 로드 완료


In [3]:

print("==> 2. 문서 분할 → 작은 청크로 나누기")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # 청크 크기 (한국어 최적화)
    chunk_overlap=200,      # 중복 부분 (맥락 보존)
    separators=["\n\n", "\n", ".", " ", ""] # 자연스러운 분할을 위한 구분자
)

chunks = text_splitter.split_documents(documents)
print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f}자")
print(type(chunks[0]))

==> 2. 문서 분할 → 작은 청크로 나누기
  66개 청크 생성 완료
  평균 청크 길이: 688자
<class 'langchain_core.documents.base.Document'>


In [4]:

# FAISS 인덱스 저장 경로 (경로 구분자는 슬래시 사용: 역슬래시는 \f 같은 이스케이프 문자로 해석될 수 있음)
DB_PATH = "../db/faiss_db_upstage"

print("==> 3. 벡터화 → 임베딩으로 변환")
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

print("==> 4. 저장 → FAISS 벡터스토어에 저장")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f" FAISS 벡터스토어 생성 완료 ({len(chunks)}개 벡터)")
# 로컬 파일로 저장
vectorstore.save_local(DB_PATH)
print(f" FAISS 인덱스 저장 완료: {DB_PATH}")

print("===> 5. 검색 → 질문과 유사한 문서 찾기")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}  # 상위 6개 관련 문서 검색
)
print(" Retriever 설정 완료")

==> 3. 벡터화 → 임베딩으로 변환
==> 4. 저장 → FAISS 벡터스토어에 저장
 FAISS 벡터스토어 생성 완료 (66개 벡터)
 FAISS 인덱스 저장 완료: ../db/faiss_db_upstage
===> 5. 검색 → 질문과 유사한 문서 찾기
 Retriever 설정 완료


In [5]:

print("===> 6. 생성 → LLM으로 답변 생성")
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

===> 6. 생성 → LLM으로 답변 생성
solar-pro3


In [6]:
def format_docs(docs):
    """문서들을 문자열로 포맷"""
    return "\n\n".join(doc.page_content for doc in docs)

prompt_template = """
당신은 BlueJ 프로그래밍 환경 전문가입니다. 
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별 설명이 필요하면 순서대로 작성하세요  
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 질문에 대한 답이 문서에 전혀 없을 때에만 "문서에서 찾을 수 없습니다"라고 답하세요. 답이 있으면 이 문구나 규칙에 대한 언급을 답변에 쓰지 마세요

답변:"""

prompt = PromptTemplate.from_template(prompt_template)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='\n당신은 BlueJ 프로그래밍 환경 전문가입니다. \n아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.\n\n문서 내용:\n{context}\n\n질문: {question}\n\n답변 규칙:\n1. 문서 내용만을 근거로 답변하세요\n2. 단계별 설명이 필요하면 순서대로 작성하세요  \n3. 구체적인 메뉴명, 버튼명을 포함하세요\n4. 질문에 대한 답이 문서에 전혀 없을 때에만 "문서에서 찾을 수 없습니다"라고 답하세요. 답이 있으면 이 문구나 규칙에 대한 언급을 답변에 쓰지 마세요\n\n답변:'


In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LCEL RAG 체인 (RetrievalQA 대체!)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 소스 문서 포함 버전
def rag_with_sources(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    
    response = llm.invoke(prompt.format(context=context, question=question))
    return {
        "answer": response.content,
        "source_documents": docs  # RetrievalQA와 동일!
    }

print("  LCEL RAG 파이프라인 구축 완료!")

  LCEL RAG 파이프라인 구축 완료!


In [8]:

# ===================================
# 테스트
# ===================================
test_questions = [
    "BlueJ에서 객체를 생성하는 방법은 무엇인가요?",
    "컴파일 오류가 발생했을 때 어떻게 확인할 수 있나요?", 
    "디버깅을 위해 중단점을 설정하는 방법을 알려주세요",
    "코드패드는 무엇이고 어떻게 사용하나요?",
    "애플릿을 만들고 실행하는 방법을 설명해주세요"
]

print("\n" + "=" * 60)
print(" LCEL RAG 시스템 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/5】")
    print(f"Q: {question}")
    
    # 답변과 소스 문서를 한 번의 검색으로 함께 얻기 (검색을 두 번 하면 임베딩 API 호출도 두 번 발생)
    result = rag_with_sources(question)
    print(f"A: {result['answer']}")

    # 소스 문서 확인 (page 는 0부터 시작하므로 문서에 표시된 페이지 번호인 page_label 사용)
    docs = result["source_documents"]
    print(f"  참조: {len(docs)}개 문서")
    for j, doc in enumerate(docs[:2], 1):
        page = doc.metadata.get('page_label', '?')
        print(f"    {j}. P{page}: {doc.page_content[:80]}...")
    
    print("-" * 40)


 LCEL RAG 시스템 테스트

【테스트 1/5】
Q: BlueJ에서 객체를 생성하는 방법은 무엇인가요?
A: BlueJ에서 객체를 생성하는 방법은 다음과 같습니다.

1. **프로젝트 열기**:  
   - BlueJ를 시작한 후 `Project - Open...` 메뉴를 선택하여 프로젝트를 엽니다.  
   - `examples` 디렉토리에 있는 `people` 프로젝트를 선택하여 엽니다.  

2. **클래스 선택**:  
   - 메인 윈도우 중앙에 표시된 클래스 아이콘(예: `Person`, `Staff`, `Student`) 중 하나를 선택합니다.  

3. **메뉴 호출**:  
   - 클래스 아이콘에서 마우스 오른쪽 버튼을 클릭(매킨토시는 `Ctrl+click`)하여 클래스 메뉴를 엽니다.  

4. **생성자 선택**:  
   - 메뉴에 표시된 `constructor` 함수를 선택합니다.  
   - 대화상자에서 생성할 객체의 이름을 입력받습니다(기본 이름 `staff_1`이 제공됨).  
   - `OK` 버튼을 클릭하면 객체가 생성됩니다.  

5. **객체 확인**:  
   - 생성된 객체는 `object bench`에 표시됩니다(예: `Staff` 객체).  

추가로, **라이브러리 클래스로부터 객체 생성** 시에는:  
- `Tools - Use Library Class` 메뉴를 선택합니다.  
- 대화상자에 완전한 클래스명(예: `java.lang.String`)을 입력하고 `Enter`를 누릅니다.  
- 해당 클래스의 생성자 또는 정적 메소드를 선택하여 객체를 생성할 수 있습니다.  

**참고**: 추상 클래스(예: `Person` 클래스 아이콘에 `<<abstract>>` 표기된 경우)는 객체를 생성할 수 없습니다.
  참조: 6개 문서
    1. P2: 3.1. BlueJ 시작하기 ············································· ··················...
    2.

### 통합 RAG 파이프라인 (단계별 코드를 하나로)

위에서 단계별로 나누어 실행한 코드를 **하나의 셀**로 합치고, 함수로 정리했습니다. 이 셀만 단독으로 실행해도 동작합니다.

#### 전체 흐름

```
PDF ─ load_chunks() ─→ 청크 ─ get_vectorstore() ─→ FAISS 인덱스 ─ as_retriever() ─→ 검색
                                                                              │
질문 ─────────────── build_rag_chain() : 검색(1회) → 프롬프트 → LLM → 답변 + 출처 ◀┘
```

| 단계 | 위의 셀 | 통합 코드 |
|---|---|---|
| 1~2. 로드, 분할 | PDF 로드(`PyPDFLoader`), 문서 분할 | `load_chunks()` (로더를 `UpstageDocumentParseLoader`로 변경) |
| 3~4. 임베딩, 저장 | 벡터화, FAISS 저장 | `get_vectorstore()` |
| 5. 검색 | Retriever 설정 | `as_retriever()` |
| 6. 생성 | LLM, 프롬프트, 체인 | `build_rag_chain()` |
| 테스트 | 질문 5개 반복 | `ask()` |

#### 개선한 점

| 개선 | 이유 |
|---|---|
| **설정을 상수로 모음** (경로, 청크 크기, `TOP_K`, 모델명) | 값을 바꿀 때 한 곳만 수정하면 됨 |
| **목차 청크 제외** (`1.1. 제목 4`처럼 번호로 시작하고 쪽수로 끝나는 줄이 절반 이상인 청크) | `PyPDFLoader` 결과에서 목차 페이지(`P2`, `P3`)가 검색 상위에 자주 나와 검색 자리를 낭비했음. Document Parse는 목차를 점선 없는 일반 텍스트로 읽으므로 줄의 형태로 판별 |
| **인덱스가 있으면 로드, 없으면 생성** | 실행할 때마다 임베딩 API를 호출하지 않아 시간과 비용을 절약 |
| **검색 1회로 답변과 출처를 함께 반환** (`RunnableParallel` + `assign`) | 같은 검색 결과로 답변하고 출처를 표시하므로 답변의 근거와 출처가 일치함 |
| **`temperature=0`** | 문서 기반 답변은 실행마다 달라지지 않게 하는 것이 좋음 |

#### 사용할 때 주의할 점
- 처음 실행하면 **Document Parse(문서 파싱)와 임베딩** 때문에 Upstage API 호출이 발생합니다. 두 번째부터는 저장된 인덱스(`DB_PATH`)를 불러옵니다.
- **로더를 바꾸면 인덱스를 새로 만들어야 합니다.** 이전 인덱스(`_v2`: `PyPDFLoader`, `_v3`: 목차 제외가 동작하지 않던 버전)와 구분하려고 `DB_PATH`를 `_v4`로 바꿨습니다.
- `UpstageDocumentParseLoader`의 `page`는 1부터 시작하는 페이지 번호이며, `PyPDFLoader`의 `page_label` 대신 이 값을 표시합니다. 파싱 결과는 인덱스에 저장되지 않아, **설정을 바꿔 다시 만들면 파싱 비용이 다시 발생**합니다.
- **청크 크기나 목차 제외 같은 설정을 바꾸면 기존 인덱스에 반영되지 않습니다.** `DB_PATH`를 바꾸거나 기존 인덱스 폴더를 삭제한 뒤 다시 실행하세요.
- `allow_dangerous_deserialization=True`는 저장 파일(`.pkl`)을 불러오기 위한 옵션이므로 **직접 만든 인덱스만** 로드하세요.
- 목차 판별은 "번호로 시작하고 쪽수로 끝나는 줄"의 비율에 의존하므로, 다른 문서에서는 `TOC_LINE_PATTERN`과 `TOC_LINE_RATIO`를 조정해야 합니다. 실행 후 `목차 N개 제외` 출력으로 동작을 확인하세요. `0개 제외`로 나오면 규칙이 맞지 않는 것입니다.

In [11]:
# ===================================
# 통합 RAG 파이프라인: PDF → 분할 → 임베딩/저장 → 검색 → 답변
# ===================================
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_upstage import ChatUpstage, UpstageDocumentParseLoader, UpstageEmbeddings

# ── 설정 ──────────────────────────────────────────
PDF_PATH = "../data/tutorial-korean.pdf"
# 설정(로더, 청크 크기, 목차 제외 등)을 바꾸면 이 경로도 바꾸거나 기존 인덱스를 삭제해야 반영됨
# (_v2: PyPDFLoader, _v3: 목차 제외가 동작하지 않던 버전 → 재사용하지 않도록 _v4 사용)
DB_PATH = "../db/faiss_db_upstage_v4"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 6                                      # 검색할 청크 수
EMBEDDING_MODEL = "solar-embedding-1-large"
CHAT_MODEL = "solar-pro3"
UPSTAGE_BASE_URL = "https://api.upstage.ai/v1"
# 목차 판별: "1.1. 제목 4"처럼 번호로 시작하고 쪽수로 끝나는 줄이 절반 이상인 청크
TOC_LINE_PATTERN = re.compile(r"^\d+(\.\d+)*\.?\s+\S.*\s\d+$")
TOC_LINE_RATIO = 0.5

PROMPT_TEMPLATE = """
당신은 BlueJ 프로그래밍 환경 전문가입니다.
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별 설명이 필요하면 순서대로 작성하세요
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 질문에 대한 답이 문서에 전혀 없을 때에만 "문서에서 찾을 수 없습니다"라고 답하세요. 답이 있으면 이 문구나 규칙에 대한 언급을 답변에 쓰지 마세요

답변:"""

TEST_QUESTIONS = [
    "BlueJ에서 객체를 생성하는 방법은 무엇인가요?",
    "컴파일 오류가 발생했을 때 어떻게 확인할 수 있나요?",
    "디버깅을 위해 중단점을 설정하는 방법을 알려주세요",
    "코드패드는 무엇이고 어떻게 사용하나요?",
    "애플릿을 만들고 실행하는 방법을 설명해주세요",
]


# ── 1~2. 로드, 분할 ───────────────────────────────
def is_toc_chunk(text: str) -> bool:
    """'번호. 제목 쪽수' 형태의 줄이 TOC_LINE_RATIO 이상이면 목차 청크로 판단합니다."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return False
    toc_lines = sum(bool(TOC_LINE_PATTERN.match(line)) for line in lines)
    return toc_lines / len(lines) >= TOC_LINE_RATIO


def load_chunks(pdf_path: str) -> list:
    """PDF를 Upstage Document Parse로 읽어 청크로 분할하고, 목차 청크는 제외해서 반환합니다."""
    # split="page": 페이지마다 Document 1개 / output_format="markdown": 표와 제목 구조를 보존
    # Upstage API 호출이 발생하므로 인덱스를 만들 때만 실행됨 (get_vectorstore 참고)
    documents = UpstageDocumentParseLoader(
        pdf_path, split="page", output_format="markdown", coordinates=False
    ).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " ", ""],  # 자연스러운 분할을 위한 구분자
    )
    chunks = splitter.split_documents(documents)
    body = [c for c in chunks if not is_toc_chunk(c.page_content)]
    print(f"  {len(documents)}페이지 → {len(chunks)}개 청크 (목차 {len(chunks) - len(body)}개 제외 → {len(body)}개)")
    return body


# ── 3~4. 임베딩, 저장 ─────────────────────────────
def get_vectorstore(embeddings, db_path: str, pdf_path: str):
    """저장된 인덱스가 있으면 로드하고, 없으면 새로 만들어 저장합니다."""
    if (Path(db_path) / "index.faiss").exists():
        print(f"  기존 인덱스 로드: {db_path}")
        # 직접 만든 인덱스만 로드할 것 (pickle 사용)
        return FAISS.load_local(db_path, embeddings, allow_dangerous_deserialization=True)
    chunks = load_chunks(pdf_path)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(db_path)
    print(f"  인덱스 생성 및 저장 완료: {db_path} ({len(chunks)}개 벡터)")
    return vectorstore


# ── 5~6. 검색 + 생성 ──────────────────────────────
def format_docs(docs) -> str:
    """검색된 문서들을 하나의 문자열로 합칩니다."""
    return "\n\n".join(doc.page_content for doc in docs)


def build_rag_chain(retriever, llm):
    """검색을 1회만 수행하고, 같은 문서로 답변 생성과 출처 반환을 모두 하는 체인을 만듭니다."""
    prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
    answer_chain = (
        RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
        | prompt
        | llm
        | StrOutputParser()
    )
    return (
        RunnableParallel(docs=retriever, question=RunnablePassthrough())  # 검색 1회
        | RunnablePassthrough.assign(answer=answer_chain)                 # 같은 검색 결과로 답변 생성
    )


def ask(chain, question: str, show_sources: int = 2) -> dict:
    """질문하고 답변과 참조 문서를 출력합니다. (page: 1부터 시작하는 페이지 번호)"""
    result = chain.invoke(question)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print(f"  참조: {len(result['docs'])}개 문서")
    for i, doc in enumerate(result["docs"][:show_sources], 1):
        snippet = doc.page_content[:80].replace("\n", " ")
        print(f"    {i}. P{doc.metadata.get('page', '?')}: {snippet}...")
    return result


In [12]:

# ── 실행 ──────────────────────────────────────────
load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")

embeddings = UpstageEmbeddings(model=EMBEDDING_MODEL)
vectorstore = get_vectorstore(embeddings, DB_PATH, PDF_PATH)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
llm = ChatUpstage(model=CHAT_MODEL, base_url=UPSTAGE_BASE_URL, temperature=0)  # 문서 기반 답변: 낮은 temperature
rag_chain = build_rag_chain(retriever, llm)

print("\n" + "=" * 60)
print(" 통합 RAG 파이프라인 테스트")
print("=" * 60)
for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f"\n【테스트 {i}/{len(TEST_QUESTIONS)}】")
    ask(rag_chain, question)
    print("-" * 40)

  39페이지 → 60개 청크 (목차 0개 제외 → 60개)
  인덱스 생성 및 저장 완료: ../db/faiss_db_upstage_v3 (60개 벡터)

 통합 RAG 파이프라인 테스트

【테스트 1/5】
Q: BlueJ에서 객체를 생성하는 방법은 무엇인가요?
A: BlueJ에서 객체를 생성하는 방법은 다음과 같습니다:

1. **클래스 아이콘에서 마우스 오른쪽 버튼 클릭**: 생성하고자 하는 클래스의 아이콘(예: Staff 클래스)을 선택한 후 마우스 오른쪽 버튼을 클릭합니다.

2. **팝업 메뉴에서 생성자 선택**: 팝업 메뉴가 나타나면, 해당 클래스의 생성자(Constructor) 함수를 선택합니다. 예를 들어 Staff 클래스의 경우 "new Staff()" 또는 "new Staff(name, yearOfBirth, roomNumber)"와 같은 생성자 옵션이 표시됩니다.

3. **매개변수 입력 (필요한 경우)**: 
   - 매개변수 없는 생성자를 선택하면, 인스턴스 이름을 입력하는 대화상자가 나타납니다(예: "staff1").
   - 매개변수 있는 생성자를 선택하면, 해당 매개변수에 대한 값을 입력하는 대화상자가 나타납니다.

4. **생성 완료**: "Ok" 버튼을 클릭하면 객체가 생성되고, 해당 객체의 인스턴스가 프로젝트 창에 표시됩니다.

이 방법은 문서에서 설명된 대로 "클래스에서 마우스 오른쪽버튼을 클릭하면 클래스 메뉴에서 클래스의 생성자 뿐만 아니라 정적 main 메소드가 포함된 메뉴를 볼 수 있습니다"라는 내용에 근거합니다.
  참조: 6개 문서
    1. P8: # 3.2. 프로젝트 열기 | 요 약 | 프로젝트를 열기 위해 Project 메뉴에서 Open을 선택하십시오. | | --- | --- |  프...
    2. P36: # 10.6. 라이브러리 클래스로부터 객체생성하기 | 요 약 | 라이브러리 클래스로부터 객체를 생성하기 위해서는, Tools - Use Libr...
--------------------------

### 결과 분석 (UpstageDocumentParseLoader 실행 결과)

**실행 조건**: `UpstageDocumentParseLoader(split="page", output_format="markdown")`, 질문 5개, 질문당 1회, `TOP_K=6`, `temperature=0`
**인덱스 결과**: 39페이지 → 60개 청크, 목차 제외 `0개` → 60개 벡터 (`faiss_db_upstage_v3`)


#### 1. PyPDFLoader와 비교 (같은 PDF, 같은 프롬프트와 설정)

| 항목 | PyPDFLoader | UpstageDocumentParseLoader |
|---|---|---|
| 청크 수 | 66개 (점선 목차 11개 제외 → 55개) | 60개 (제외 0개 → 60개) |
| 제목과 페이지 번호 | `83.2. 프로젝트 열기`, `3610.6.`처럼 **페이지 번호가 본문에 붙음** | `# 3.2. 프로젝트 열기`, `# 10.6.`처럼 **깨끗한 마크다운 제목** |
| 표 | 평문으로 풀림 | 마크다운 표 (`\| 요 약 \| ... \|`) |
| 그림 속 글자 | 추출 불가 | **추출됨** (Q5의 `Height - 250`, `appletviewer`는 P32 그림 속 글자) |
| 목차 표현 | 점선(`····`)이 있는 텍스트 | 점선 없는 일반 텍스트 (`1. 서문 4`) |
| 페이지 번호 | `page_label` | `page` (1부터 시작, `P8`, `P36`, `P21`이 `PyPDFLoader`와 같은 값) |

두 실행은 프롬프트, `temperature`, `TOP_K`가 같으므로 답변 차이는 주로 로더(와 목차 청크 포함 여부)에서 비롯되었을 가능성이 큽니다.

#### 2. 답변 분석

| 질문 | 평가 |
|---|---|
| Q1 객체 생성 | 기본 방법(클래스 우클릭 → 생성자 선택)을 **먼저** 안내했습니다. `PyPDFLoader` 통합 실행에서는 라이브러리 클래스 방식이 먼저 나왔던 점이 개선되었습니다. 다만 "객체가 프로젝트 창에 표시된다"는 원문과 다릅니다(원문은 오브젝트 벤치). |
| Q2 컴파일 오류 | 정보창 메시지, 물음표 도움말, 오브젝트 벤치 객체 소멸 등 이전 실행과 같은 핵심을 포함했습니다. |
| Q3 중단점 | 8단계로 설정부터 제거 방법까지 순서대로 설명했습니다. |
| Q4 코드패드 | 내용은 문서(요약 페이지 P37)에 근거하지만 **한 문단으로 길게 이어져** 가독성이 낮습니다. 프롬프트의 "단계별 설명이 필요하면 순서대로 작성" 규칙이 지켜지지 않았습니다. |
| Q5 애플릿 | 그림 속 글자까지 반영해 상세하지만 매우 깁니다. "방법 2"가 "위와 동일"이라는 반복 설명입니다. |

**원문 근거 확인**
- `new Staff(), new Staff(name, yearOfBirth, roomNumber)`와 "클래스 메뉴에서 ... 정적 main 메소드가 포함된 메뉴" 인용문은 PDF 원문에 있습니다. (Q1)
- `Height - 250`, `Run Applet in appletviewer`는 P32 청크에 있으며, `PyPDFLoader`의 텍스트에는 없는 **그림 속 글자**입니다. (Q5)
- 규칙 문구("문서에서 찾을 수 없습니다...")가 답변에 나오는 문제는 5개 모두 없었습니다.

#### 3. 목차 제외가 동작하지 않은 원인과 수정

- **원인**: Document Parse는 목차를 점선 없는 일반 텍스트(`1. 서문 4`, `1.1. BlueJ의 소개 4`)로 읽는데, 기존 규칙은 점선(`····`)을 찾았습니다. 그래서 2~3페이지의 목차 청크 2개가 인덱스에 남았습니다.
- **수정**: "번호로 시작하고 쪽수로 끝나는 줄"의 비율이 0.5 이상이면 목차로 판단하는 `is_toc_chunk()`로 교체했습니다. 저장된 인덱스에서 확인했을 때 목차 청크는 비율이 **1.0**, 본문 청크는 최대 **0.06**이어서 임계값 0.5~0.7에서 정확히 2개(P2, P3)만 선택되었습니다.
- 수정 후 결과(`목차 2개 제외 → 58개` 예상)는 아직 실행으로 확인하지 않았습니다.

#### 4. 결과를 해석할 때의 한계

- 질문 5개를 **1회씩** 실행한 결과이므로 경향을 보는 용도입니다.
- 목차 청크가 인덱스에 남아 있었으므로 **3~6번째 참조 문서에 목차가 포함되었는지는 확인하지 못했습니다.** (출력된 상위 2개에는 목차가 없었습니다.)
- Document Parse는 API 비용이 발생하며, 파싱 결과는 인덱스에 저장되지 않아 인덱스를 다시 만들 때마다 파싱이 반복됩니다.

#### 5. 다음에 해볼 것

1. 수정된 코드를 실행해 `목차 2개 제외` 출력과 답변 변화를 확인
2. Q4, Q5처럼 서식이 무너지거나 장황한 답변은 프롬프트에 "항목은 번호 목록으로 구분, 질문과 직접 관련된 내용만" 같은 규칙을 추가해 비교
3. 파싱 결과를 파일로 저장해 두면 청크 설정을 바꿔 실험할 때 파싱 비용을 아낄 수 있음